In [ ]:
pip install gotabpfn

In [ ]:
pip show gotabpfn

# Dummy example with binary classification

In [ ]:
# ============================================================
# GOTabPFN Dummy End-to-End Test Script
# Creates dummy HDLSS data, learns GO-LR ordering, applies NSC,
# and evaluates frozen TabPFN-2.5 head with repeated CV.
# Includes deltas for transition-aware NSC segmentation.
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from sklearn.datasets import make_classification
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config


# ============================================================
# User settings
# ============================================================

SEED = 42
DATA_FILE = "dummy_hdlss_encoded.csv"
TARGET_COL = "label"

# Dummy HDLSS-style dataset settings
N_SAMPLES = 80
N_FEATURES = 2000
N_INFORMATIVE = 80
N_REDUNDANT = 40
N_CLASSES = 2

# Quick smoke-test CV settings
# For full Colon-style 5x5 CV, set N_SPLITS=5 and N_REPEATS=5.
N_SPLITS = 3
N_REPEATS = 1

# GO-LR settings
GO_METRIC = "euclidean"
GO_NUM_CLUSTERS = 5          # use 10 for Colon-style full setting
GO_REFINE_PASSES = 1         # use 3 for Colon-style full setting
GO_DIRECTION_SELECT = True

# NSC settings
# equal_mass and largest_jump require deltas in nsc.configure(...)
NSC_SEGMENTATION = "equal_mass"  # options: "uniform", "equal_mass", "largest_jump"
NSC_M_RULE = "idf"
NSC_TAU = 0.99
NSC_GAMMA = 1.7570143129240916
NSC_BETA = 0.2244046472232107
NSC_MMIN = 16                # use 64 for Colon-style full setting
NSC_MMAX = 64                # use 384 for Colon-style full setting
NSC_LMIN = 8                 # use 16 for Colon-style full setting
ASSUME_STANDARDIZED = False

# TabPFN settings
TABPFN_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")


# ============================================================
# Helper functions
# ============================================================

def compute_adjacent_corr_deltas(X, Pi_star, eps=1e-8):
    """
    Compute transition-aware deltas along the learned feature order.

    deltas[t] = 1 - |corr(feature Pi_star[t], feature Pi_star[t+1])|

    Shape:
        (m - 1,)

    Used by transition-aware NSC segmentation methods:
        - equal_mass
        - largest_jump
    """
    Xo = X[:, Pi_star].astype(np.float64)

    # Standardize columns for correlation computation
    Xo = Xo - Xo.mean(axis=0, keepdims=True)
    Xo = Xo / (Xo.std(axis=0, keepdims=True) + eps)

    a = Xo[:, :-1]
    b = Xo[:, 1:]

    corr_adj = np.mean(a * b, axis=0)
    deltas = 1.0 - np.abs(corr_adj)

    return deltas.astype(np.float32)


def safe_auc(y_true, proba, num_classes):
    """
    Compute binary or multiclass ROC-AUC safely.
    Returns np.nan if AUC cannot be computed for a fold.
    """
    try:
        if num_classes == 2:
            return roc_auc_score(y_true, proba[:, 1])
        return roc_auc_score(y_true, proba, multi_class="ovr", average="macro")
    except Exception:
        return np.nan


# ============================================================
# Create dummy HDLSS-style dataset and save as CSV
# ============================================================

print("\nCreating dummy HDLSS dataset...")

X_dummy, y_dummy = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=N_INFORMATIVE,
    n_redundant=N_REDUNDANT,
    n_repeated=0,
    n_classes=N_CLASSES,
    class_sep=1.2,
    flip_y=0.03,
    random_state=SEED,
)

feature_cols = [f"f{i}" for i in range(N_FEATURES)]
df_dummy = pd.DataFrame(X_dummy, columns=feature_cols)
df_dummy[TARGET_COL] = y_dummy
df_dummy.to_csv(DATA_FILE, index=False)

print(f"Saved dummy dataset to: {DATA_FILE}")
print(f"Dataset shape: {df_dummy.shape}")
print("Class counts:")
print(df_dummy[TARGET_COL].value_counts())


# ============================================================
# Load and preprocess data
# ============================================================

print("\nLoading and preprocessing data...")

df = pd.read_csv(DATA_FILE)

if TARGET_COL not in df.columns:
    raise ValueError(f"TARGET_COL='{TARGET_COL}' not found in CSV columns.")

y_raw = df[TARGET_COL].astype(str).fillna("missing_target")
X_df = df.drop(columns=[TARGET_COL])

# Keep numeric features only
X_df = X_df.select_dtypes(include=[np.number])
X_df = X_df.apply(pd.to_numeric, errors="coerce")
X_df = X_df.fillna(X_df.median(numeric_only=True)).fillna(0.0)

if X_df.shape[1] == 0:
    raise ValueError("No numeric feature columns found after preprocessing.")

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y_raw).astype(np.int64)
num_classes = len(np.unique(y))

if num_classes < 2:
    raise ValueError("Need at least two classes for classification.")

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X_df.values).astype(np.float32)

print(f"X shape: {X.shape}")
print(f"Number of classes: {num_classes}")
print(f"Classes: {list(le.classes_)}")


# ============================================================
# Learn GO-LR feature ordering once
# ============================================================

print("\nLearning GO-LR feature ordering...")

go = GraphFeatureOrdering(
    num_clusters=GO_NUM_CLUSTERS,
    metric=GO_METRIC,
    refine=True,
    direction_select=GO_DIRECTION_SELECT,
    refine_passes=GO_REFINE_PASSES,
)

try:
    Pi_star, _, _, _ = go.fit(
        X,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=False,
    )
except Exception as e:
    print("GPU/standard k-means failed; retrying with CPU k-means.")
    print(f"Reason: {repr(e)}")
    Pi_star, _, _, _ = go.fit(
        X,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=True,
    )

Pi_star = list(map(int, Pi_star))

print(f"Learned feature ordering length: {len(Pi_star)}")
print(f"First 10 ordered feature indices: {Pi_star[:10]}")


# ============================================================
# Compute transition deltas if needed
# ============================================================

transition_aware_segmentations = {"equal_mass", "largest_jump"}

if NSC_SEGMENTATION in transition_aware_segmentations:
    print(f"\nComputing deltas for transition-aware segmentation: {NSC_SEGMENTATION}")
    deltas = compute_adjacent_corr_deltas(X, Pi_star)
    print(f"Deltas shape: {deltas.shape}")
    print(f"Deltas min/mean/max: {deltas.min():.4f} / {deltas.mean():.4f} / {deltas.max():.4f}")
else:
    deltas = None
    print("\nUniform segmentation selected; deltas not required.")


# ============================================================
# Cross-validation setup
# ============================================================

rkf = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=SEED,
)

head_cfg = TabPFN25Config(
    task_type="binary" if num_classes == 2 else "multiclass",
    num_classes=num_classes,
    device=DEVICE,
    random_state=TABPFN_SEED,
)

accs, f1s, aucs = [], [], []


# ============================================================
# CV evaluation
# ============================================================

print("\nStarting cross-validation...")

for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X, y), start=1):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    nsc = pidf_segpca(
        segmentation=NSC_SEGMENTATION,
        l_min=NSC_LMIN,
        m_rule=NSC_M_RULE,
        gamma=NSC_GAMMA,
        beta=NSC_BETA,
        tau=NSC_TAU,
        M_min=NSC_MMIN,
        M_max=NSC_MMAX,
        assume_standardized=ASSUME_STANDARDIZED,
        device=DEVICE,
    )

    X_tr_t = torch.from_numpy(X_tr)

    configure_kwargs = {
        "Pi_star": Pi_star,
        "X_train": X_tr_t,
        "tau": NSC_TAU,
    }

    if NSC_SEGMENTATION in transition_aware_segmentations:
        configure_kwargs["deltas"] = deltas

    nsc.configure(**configure_kwargs)

    Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
    Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

    head = TabPFN25Head(head_cfg)
    head.fit(Z_tr, y_tr)

    P = head.predict_proba(Z_va)
    pred = np.argmax(P, axis=1)

    acc = accuracy_score(y_va, pred)
    f1 = f1_score(y_va, pred, average="macro")
    auc = safe_auc(y_va, P, num_classes)

    accs.append(acc)
    f1s.append(f1)
    aucs.append(auc)

    print(
        f"Fold {fold_id:02d}: "
        f"Z_tr={Z_tr.shape}, Z_va={Z_va.shape}, "
        f"ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}"
    )


# ============================================================
# Final results
# ============================================================

accs = np.asarray(accs, dtype=float)
f1s = np.asarray(f1s, dtype=float)
aucs = np.asarray(aucs, dtype=float)

print("\nFinal CV results")
print(f"Accuracy : {np.nanmean(accs):.4f} ± {np.nanstd(accs, ddof=1):.4f}")
print(f"Macro-F1 : {np.nanmean(f1s):.4f} ± {np.nanstd(f1s, ddof=1):.4f}")
print(f"AUC      : {np.nanmean(aucs):.4f} ± {np.nanstd(aucs, ddof=1):.4f}")


# ============================================================
# Optional: full Colon-style settings
# ============================================================
# To run a heavier Colon-style test, change:
#
# N_SPLITS = 5
# N_REPEATS = 5
# GO_NUM_CLUSTERS = 10
# GO_REFINE_PASSES = 3
# NSC_MMIN = 64
# NSC_MMAX = 384
# NSC_LMIN = 16
# NSC_SEGMENTATION = "equal_mass"
#
# Then rerun the whole script.
# ============================================================

# Dummy multiclass classification example

In [ ]:
# ============================================================
# GOTabPFN Dummy Multiclass End-to-End Test
# Creates ORL-like multiclass high-dimensional data,
# learns GO-LR ordering, applies NSC, and evaluates TabPFN-2.5.
# ============================================================

import gc
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch

from sklearn.datasets import make_classification
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config

warnings.filterwarnings("ignore")


# ============================================================
# User settings
# ============================================================

DATA_FILE = "dummy_orlraws10P.csv"
TARGET_COL = "label"
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Dummy ORL-like multiclass HDLSS settings
# ORL-like: small n, many features, 10 classes.
N_SAMPLES = 100
N_FEATURES = 3000      # use 10304 if you want closer ORL-like dimensionality
N_CLASSES = 10
N_INFORMATIVE = 250
N_REDUNDANT = 100

# Quick smoke-test CV
# Use 5 and 5 for full 5x5 CV.
N_SPLITS = 5
N_REPEATS = 1

# Fixed GOTabPFN hyperparameters, ORL-style
FIXED_PARAMS = {
    "go_metric": "cosine",
    "go_num_clusters": 5,
    "go_refine_passes": 1,
    "go_direction_select": False,
    "go_feat_subsample": 3000,

    "nsc_segmentation": "uniform",
    "nsc_m_rule": "default",
    "nsc_tau": 0.99,
    "nsc_gamma": 2.049512863264476,
    "nsc_beta": 0.3887505167779042,
    "nsc_Mmin": 32,
    "nsc_Mmax": 384,
    "nsc_lmin": 12,
    "assume_standardized": False,

    "tabpfn_seed": 42,
}


# ============================================================
# Utilities
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_multiclass_macro_ovr_auc(y_true, proba, num_classes):
    """
    Macro OvR AUC for multiclass classification.
    Returns np.nan if one class is missing in a validation fold.
    """
    try:
        y_bin = label_binarize(y_true, classes=np.arange(num_classes))
        return float(
            roc_auc_score(
                y_bin,
                proba,
                average="macro",
                multi_class="ovr",
            )
        )
    except Exception:
        return np.nan


# ============================================================
# Create dummy multiclass high-dimensional dataset
# ============================================================

seed_everything(SEED)

print("\nCreating dummy multiclass HDLSS dataset...")

X_dummy, y_dummy = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=N_INFORMATIVE,
    n_redundant=N_REDUNDANT,
    n_repeated=0,
    n_classes=N_CLASSES,
    n_clusters_per_class=1,
    class_sep=1.5,
    flip_y=0.01,
    random_state=SEED,
)

# Make labels string-like to test LabelEncoder, similar to real CSV workflows
label_names = np.array([f"class_{i}" for i in range(N_CLASSES)])
y_labels = label_names[y_dummy]

feature_cols = [f"f{i}" for i in range(N_FEATURES)]
df_dummy = pd.DataFrame(X_dummy, columns=feature_cols)
df_dummy[TARGET_COL] = y_labels
df_dummy.to_csv(DATA_FILE, index=False)

print(f"Saved dummy dataset to: {DATA_FILE}")
print(f"Dataset shape: {df_dummy.shape}")
print("Class counts:")
print(df_dummy[TARGET_COL].value_counts().sort_index())


# ============================================================
# Load and preprocess data
# ============================================================

print("\nLoading and preprocessing data...")

df = pd.read_csv(DATA_FILE)

if TARGET_COL not in df.columns:
    raise ValueError(f"TARGET_COL='{TARGET_COL}' not found in CSV columns.")

y_raw = df[TARGET_COL].astype(str).fillna("missing_target")
X_df = df.drop(columns=[TARGET_COL])

# Keep numeric features only
X_df = X_df.select_dtypes(include=[np.number])
X_df = X_df.apply(pd.to_numeric, errors="coerce")
X_df = X_df.fillna(X_df.median(numeric_only=True)).fillna(0.0)

if X_df.shape[1] == 0:
    raise ValueError("No numeric feature columns found after preprocessing.")

# Encode multiclass labels
le = LabelEncoder()
y = le.fit_transform(y_raw).astype(np.int64)
num_classes = len(le.classes_)

if num_classes < 2:
    raise ValueError("Need at least two classes for classification.")

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X_df.values).astype(np.float32)

print(f"X shape: {X.shape}, classes: {num_classes}")
print(f"Encoded classes: {list(le.classes_)}")


# ============================================================
# Learn GO-LR ordering once
# ============================================================

print("\nLearning GO-LR feature ordering...")

m_full = X.shape[1]
feat_subsample = FIXED_PARAMS["go_feat_subsample"]

rng = np.random.default_rng(SEED + 999)

if feat_subsample is not None and feat_subsample < m_full:
    feat_idx = rng.choice(m_full, size=feat_subsample, replace=False)
    feat_idx.sort()
else:
    feat_idx = np.arange(m_full)

X_go = X[:, feat_idx]

print(f"Full feature count: {m_full}")
print(f"GO-LR feature subset size: {len(feat_idx)}")

go = GraphFeatureOrdering(
    num_clusters=int(FIXED_PARAMS["go_num_clusters"]),
    metric=FIXED_PARAMS["go_metric"],
    refine=True,
    direction_select=bool(FIXED_PARAMS["go_direction_select"]),
    refine_passes=int(FIXED_PARAMS["go_refine_passes"]),
)

try:
    Pi_sub, _, _, _ = go.fit(
        X_go,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=False,
    )
except Exception as e:
    print("GPU/standard k-means failed; retrying with CPU k-means.")
    print(f"Reason: {repr(e)}")
    cleanup_cuda()
    Pi_sub, _, _, _ = go.fit(
        X_go,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=True,
    )

ordered_subset = feat_idx[np.array(Pi_sub, dtype=np.int64)].tolist()

if len(feat_idx) < m_full:
    remaining = np.setdiff1d(np.arange(m_full), feat_idx, assume_unique=False)
    Pi_star = ordered_subset + remaining.tolist()
else:
    Pi_star = ordered_subset

Pi_star = list(map(int, Pi_star))

print(f"Learned global ordering length: {len(Pi_star)}")
print(f"First 10 ordered feature indices: {Pi_star[:10]}")


# ============================================================
# 5x repeated stratified cross-validation
# ============================================================

rkf = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=SEED,
)

head_cfg = TabPFN25Config(
    task_type="multiclass",
    num_classes=int(num_classes),
    device=DEVICE,
    random_state=int(FIXED_PARAMS["tabpfn_seed"]),
)

accs, f1s, aucs = [], [], []
t0 = time.perf_counter()

print("\nStarting cross-validation...")

for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X, y), start=1):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    nsc = pidf_segpca(
        segmentation=FIXED_PARAMS["nsc_segmentation"],
        l_min=int(FIXED_PARAMS["nsc_lmin"]),
        m_rule=FIXED_PARAMS["nsc_m_rule"],
        gamma=float(FIXED_PARAMS["nsc_gamma"]),
        beta=float(FIXED_PARAMS["nsc_beta"]),
        tau=float(FIXED_PARAMS["nsc_tau"]),
        M_min=int(FIXED_PARAMS["nsc_Mmin"]),
        M_max=int(FIXED_PARAMS["nsc_Mmax"]),
        assume_standardized=bool(FIXED_PARAMS["assume_standardized"]),
        device=DEVICE,
    )

    X_tr_t = torch.from_numpy(X_tr)

    # For uniform segmentation, deltas are not needed.
    # For equal_mass/largest_jump, pass deltas.
    nsc.configure(
        Pi_star=Pi_star,
        X_train=X_tr_t,
        tau=float(FIXED_PARAMS["nsc_tau"]),
        deltas=None,
    )

    Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
    Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

    head = TabPFN25Head(head_cfg)
    head.fit(Z_tr, y_tr)

    P = head.predict_proba(Z_va)

    # Safety: normalize if needed
    P = np.asarray(P)
    if P.ndim != 2 or P.shape[1] != num_classes:
        raise ValueError(f"Unexpected probability shape: {P.shape}")

    pred = np.argmax(P, axis=1)

    acc = float(accuracy_score(y_va, pred))
    f1m = float(f1_score(y_va, pred, average="macro"))
    aucm = safe_multiclass_macro_ovr_auc(y_va, P, num_classes)

    accs.append(acc)
    f1s.append(f1m)
    aucs.append(aucm)

    print(
        f"Fold {fold_id:02d}: "
        f"Z_tr={Z_tr.shape}, Z_va={Z_va.shape}, "
        f"ACC={acc:.4f}, Macro-F1={f1m:.4f}, Macro-OvR-AUC={aucm:.4f}"
    )

    cleanup_cuda()


# ============================================================
# Final results
# ============================================================

accs = np.asarray(accs, dtype=float)
f1s = np.asarray(f1s, dtype=float)
aucs = np.asarray(aucs, dtype=float)

print("\nFinal CV results")
print(f"Accuracy      : {np.nanmean(accs):.4f} ± {np.nanstd(accs, ddof=1):.4f}")
print(f"Macro-F1      : {np.nanmean(f1s):.4f} ± {np.nanstd(f1s, ddof=1):.4f}")
print(f"Macro-OvR-AUC : {np.nanmean(aucs):.4f} ± {np.nanstd(aucs, ddof=1):.4f}")
print(f"Elapsed time  : {time.perf_counter() - t0:.2f} seconds")


# ============================================================
# Optional closer ORL-like setting
# ============================================================
# To make it closer to orlraws10P:
#
# N_FEATURES = 10304
# FIXED_PARAMS["go_feat_subsample"] = 3000
# N_SPLITS = 5
# N_REPEATS = 5
#
# Then rerun the whole script.
# ============================================================

# Dummy regression example

In [ ]:
# ============================================================
# GOTabPFN Dummy Regression End-to-End Test
# Creates DrivFace-like high-dimensional regression data,
# learns GO-LR ordering, applies NSC, and evaluates TabPFN-2.5.
# ============================================================

import gc
import time
import random
import warnings

import numpy as np
import pandas as pd
import torch

from sklearn.datasets import make_regression
from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config

warnings.filterwarnings("ignore")


# ============================================================
# User settings
# ============================================================

DATA_FILE = "dummy_drivface_regression.csv"
TARGET_COL = "angle"
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# Dummy DrivFace-like regression setting
# Real DrivFace-style example: n around 606, m around 6400.
N_SAMPLES = 300
N_FEATURES = 3000      # use 6400 for closer DrivFace-like dimensionality
N_INFORMATIVE = 150
NOISE = 15.0

# Quick smoke-test CV
# Use 5 and 5 for full 5x5 CV.
N_SPLITS = 5
N_REPEATS = 1

# Fixed GOTabPFN hyperparameters, DrivFace-style
FIXED_PARAMS = {
    "go_metric": "manhattan",
    "go_num_clusters": 5,
    "go_refine_passes": 1,
    "go_direction_select": False,
    "go_feat_subsample": 2000,

    "nsc_segmentation": "largest_jump",
    "nsc_m_rule": "idf",
    "nsc_tau": 0.99,
    "nsc_gamma": 2.654390393837633,
    "nsc_beta": 0.043192175152615336,
    "nsc_Mmin": 16,
    "nsc_Mmax": 256,
    "nsc_lmin": 12,
    "assume_standardized": True,

    "tabpfn_seed": 3,
}


# ============================================================
# Utilities
# ============================================================

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def compute_deltas_adjacent_corr(X_tr, Pi_star, eps=1e-12):
    """
    Compute transition-aware deltas along the learned feature order.

    deltas[t] = 1 - |corr(feature Pi_star[t], feature Pi_star[t+1])|

    Shape:
        (m - 1,)

    Used by transition-aware NSC segmentation methods:
        - equal_mass
        - largest_jump
    """
    X_t = torch.from_numpy(X_tr).float()
    perm = torch.tensor(Pi_star, dtype=torch.long)

    Xp = X_t[:, perm]
    Xc = Xp - Xp.mean(dim=0, keepdim=True)
    std = Xc.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z = Xc / std

    corr = (Z[:, :-1] * Z[:, 1:]).mean(dim=0)
    return (1.0 - corr.abs()).cpu()


# ============================================================
# Create dummy high-dimensional regression dataset
# ============================================================

seed_everything(SEED)

print("\nCreating dummy high-dimensional regression dataset...")

X_dummy, y_dummy = make_regression(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=N_INFORMATIVE,
    noise=NOISE,
    random_state=SEED,
)

# Rescale target to look like an angle-style regression target
# This is only for a realistic-looking dummy target.
y_dummy = StandardScaler().fit_transform(y_dummy.reshape(-1, 1)).reshape(-1)
y_dummy = 45.0 + 15.0 * y_dummy

feature_cols = [f"f{i}" for i in range(N_FEATURES)]
df_dummy = pd.DataFrame(X_dummy, columns=feature_cols)
df_dummy[TARGET_COL] = y_dummy.astype(np.float32)
df_dummy.to_csv(DATA_FILE, index=False)

print(f"Saved dummy dataset to: {DATA_FILE}")
print(f"Dataset shape: {df_dummy.shape}")
print(df_dummy[TARGET_COL].describe())


# ============================================================
# Load and preprocess data
# ============================================================

print("\nLoading and preprocessing data...")

df = pd.read_csv(DATA_FILE)

if TARGET_COL not in df.columns:
    raise ValueError(f"TARGET_COL='{TARGET_COL}' not found in CSV columns.")

# Regression target
y_raw = pd.to_numeric(df[TARGET_COL], errors="coerce")
y_raw = y_raw.fillna(y_raw.median())
y = y_raw.to_numpy(dtype=np.float32)

X_df = df.drop(columns=[TARGET_COL])

# Keep numeric features only
X_df = X_df.select_dtypes(include=[np.number])
X_df = X_df.apply(pd.to_numeric, errors="coerce")
X_df = X_df.fillna(X_df.median(numeric_only=True)).fillna(0.0)

if X_df.shape[1] == 0:
    raise ValueError("No numeric feature columns found after preprocessing.")

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X_df.values).astype(np.float32)

print(f"X shape: {X.shape}, y shape: {y.shape}")


# ============================================================
# Learn GO-LR ordering once
# ============================================================

print("\nLearning GO-LR feature ordering...")

m_full = X.shape[1]
feat_subsample = FIXED_PARAMS["go_feat_subsample"]

rng = np.random.default_rng(SEED + 999)

if feat_subsample is not None and feat_subsample < m_full:
    feat_idx = rng.choice(m_full, size=feat_subsample, replace=False)
    feat_idx.sort()
else:
    feat_idx = np.arange(m_full)

X_go = X[:, feat_idx]

print(f"Full feature count: {m_full}")
print(f"GO-LR feature subset size: {len(feat_idx)}")

go = GraphFeatureOrdering(
    num_clusters=int(FIXED_PARAMS["go_num_clusters"]),
    metric=FIXED_PARAMS["go_metric"],
    refine=True,
    direction_select=bool(FIXED_PARAMS["go_direction_select"]),
    refine_passes=int(FIXED_PARAMS["go_refine_passes"]),
)

try:
    Pi_sub, _, _, _ = go.fit(
        X_go,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=False,
    )
except Exception as e:
    print("GPU/standard k-means failed; retrying with CPU k-means.")
    print(f"Reason: {repr(e)}")
    cleanup_cuda()
    Pi_sub, _, _, _ = go.fit(
        X_go,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=True,
    )

ordered_subset = feat_idx[np.array(Pi_sub, dtype=np.int64)].tolist()

if len(feat_idx) < m_full:
    remaining = np.setdiff1d(np.arange(m_full), feat_idx, assume_unique=False)
    Pi_star = ordered_subset + remaining.tolist()
else:
    Pi_star = ordered_subset

Pi_star = list(map(int, Pi_star))

print(f"Learned global ordering length: {len(Pi_star)}")
print(f"First 10 ordered feature indices: {Pi_star[:10]}")


# ============================================================
# Repeated K-Fold CV for regression
# ============================================================

rkf = RepeatedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=SEED,
)

head_cfg = TabPFN25Config(
    task_type="regression",
    num_classes=1,
    device=DEVICE,
    random_state=int(FIXED_PARAMS["tabpfn_seed"]),
)

r2s, rmses, maes = [], [], []
t0 = time.perf_counter()

print("\nStarting cross-validation...")

for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X), start=1):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    nsc = pidf_segpca(
        segmentation=FIXED_PARAMS["nsc_segmentation"],
        l_min=int(FIXED_PARAMS["nsc_lmin"]),
        m_rule=FIXED_PARAMS["nsc_m_rule"],
        gamma=float(FIXED_PARAMS["nsc_gamma"]),
        beta=float(FIXED_PARAMS["nsc_beta"]),
        tau=float(FIXED_PARAMS["nsc_tau"]),
        M_min=int(FIXED_PARAMS["nsc_Mmin"]),
        M_max=int(FIXED_PARAMS["nsc_Mmax"]),
        assume_standardized=bool(FIXED_PARAMS["assume_standardized"]),
        device=DEVICE,
    )

    deltas = None
    if FIXED_PARAMS["nsc_segmentation"] != "uniform":
        deltas = compute_deltas_adjacent_corr(X_tr, Pi_star)

    X_tr_t = torch.from_numpy(X_tr)

    nsc.configure(
        Pi_star=Pi_star,
        X_train=X_tr_t,
        tau=float(FIXED_PARAMS["nsc_tau"]),
        deltas=deltas,
    )

    Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
    Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

    head = TabPFN25Head(head_cfg)
    head.fit(Z_tr, y_tr)

    pred = np.asarray(head.predict(Z_va), dtype=np.float32).reshape(-1)

    r2 = float(r2_score(y_va, pred))
    rmse = float(np.sqrt(mean_squared_error(y_va, pred)))
    mae = float(mean_absolute_error(y_va, pred))

    r2s.append(r2)
    rmses.append(rmse)
    maes.append(mae)

    print(
        f"Fold {fold_id:02d}: "
        f"Z_tr={Z_tr.shape}, Z_va={Z_va.shape}, "
        f"R2={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}"
    )

    cleanup_cuda()


# ============================================================
# Final results
# ============================================================

r2s = np.asarray(r2s, dtype=float)
rmses = np.asarray(rmses, dtype=float)
maes = np.asarray(maes, dtype=float)

print("\nFinal CV results")
print(f"R2   : {np.nanmean(r2s):.4f} ± {np.nanstd(r2s, ddof=1):.4f}")
print(f"RMSE : {np.nanmean(rmses):.4f} ± {np.nanstd(rmses, ddof=1):.4f}")
print(f"MAE  : {np.nanmean(maes):.4f} ± {np.nanstd(maes, ddof=1):.4f}")
print(f"Elapsed time: {time.perf_counter() - t0:.2f} seconds")


# ============================================================
# Optional closer DrivFace-like setting
# ============================================================
# To make it closer to your DrivFace regression setup:
#
# N_SAMPLES = 606
# N_FEATURES = 6400
# FIXED_PARAMS["go_feat_subsample"] = 2000
# N_SPLITS = 5
# N_REPEATS = 5
#
# Then rerun the whole script.
# ============================================================

# Binary classification test with Colon data

In [ ]:
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config


# -----------------------
# User settings
# -----------------------
DATA_FILE = "coloncancer_encoded.csv"  # change your dataset file name
TARGET_COL = "label"                   # change your dataset target column
SEED = 42

# Fixed GOTabPFN hyperparameters
GO_METRIC = "euclidean"
GO_NUM_CLUSTERS = 10
GO_REFINE_PASSES = 3
GO_DIRECTION_SELECT = True

NSC_SEGMENTATION = "equal_mass"
NSC_M_RULE = "idf"
NSC_TAU = 0.99
NSC_GAMMA = 1.7570143129240916
NSC_BETA = 0.2244046472232107
NSC_MMIN = 64
NSC_MMAX = 384
NSC_LMIN = 16
ASSUME_STANDARDIZED = False

TABPFN_SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# -----------------------
# Utility
# -----------------------
def compute_deltas_adjacent_corr(X_tr, Pi_star, eps=1e-12):
    """
    Compute adjacent transition scores along the GO-LR order:
        delta_t = 1 - |corr(feature_t, feature_{t+1})|.

    Required for transition-aware NSC segmentation rules:
        - equal_mass
        - largest_jump
    """
    X_t = torch.from_numpy(X_tr).float()
    perm = torch.tensor(Pi_star, dtype=torch.long)

    Xp = X_t[:, perm]
    Xc = Xp - Xp.mean(dim=0, keepdim=True)
    std = Xc.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z = Xc / std

    corr_adj = (Z[:, :-1] * Z[:, 1:]).mean(dim=0)
    deltas = 1.0 - corr_adj.abs()

    return deltas.cpu()


# -----------------------
# Load and preprocess data
# -----------------------
df = pd.read_csv(DATA_FILE)

if TARGET_COL not in df.columns:
    raise ValueError(f"TARGET_COL='{TARGET_COL}' not found in the CSV file.")

y_raw = df[TARGET_COL].astype(str).fillna("missing_target")
X_df = df.drop(columns=[TARGET_COL])

# Keep numeric features only
X_df = X_df.select_dtypes(include=[np.number])
X_df = X_df.apply(pd.to_numeric, errors="coerce")
X_df = X_df.fillna(X_df.median(numeric_only=True)).fillna(0.0)

if X_df.shape[1] == 0:
    raise ValueError("No numeric feature columns found after preprocessing.")

# Encode labels
le = LabelEncoder()
y = le.fit_transform(y_raw).astype(np.int64)

num_classes = len(le.classes_)
if num_classes != 2:
    raise ValueError(
        f"This example expects binary classification, but found {num_classes} classes."
    )

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X_df.values).astype(np.float32)

print(f"X shape: {X.shape}")
print(f"Classes: {list(le.classes_)}")
print(f"Using device: {DEVICE}")


# -----------------------
# Learn GO-LR feature ordering once
# -----------------------
go = GraphFeatureOrdering(
    num_clusters=GO_NUM_CLUSTERS,
    metric=GO_METRIC,
    refine=True,
    direction_select=GO_DIRECTION_SELECT,
    refine_passes=GO_REFINE_PASSES,
)

try:
    Pi_star, _, _, _ = go.fit(
        X,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=False,
    )
except Exception:
    Pi_star, _, _, _ = go.fit(
        X,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=True,
    )

Pi_star = list(map(int, Pi_star))

print(f"Learned GO-LR order length: {len(Pi_star)}")


# -----------------------
# 5x5 cross-validation
# -----------------------
rkf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=SEED,
)

head_cfg = TabPFN25Config(
    task_type="binary",
    num_classes=2,
    device=DEVICE,
    random_state=TABPFN_SEED,
)

accs, f1s, aucs = [], [], []

for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X, y), start=1):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    nsc = pidf_segpca(
        segmentation=NSC_SEGMENTATION,
        l_min=NSC_LMIN,
        m_rule=NSC_M_RULE,
        gamma=NSC_GAMMA,
        beta=NSC_BETA,
        tau=NSC_TAU,
        M_min=NSC_MMIN,
        M_max=NSC_MMAX,
        assume_standardized=ASSUME_STANDARDIZED,
        device=DEVICE,
    )

    X_tr_t = torch.from_numpy(X_tr)

    # equal_mass and largest_jump require transition scores.
    deltas = None
    if NSC_SEGMENTATION in {"equal_mass", "largest_jump"}:
        deltas = compute_deltas_adjacent_corr(X_tr, Pi_star)

    nsc.configure(
        Pi_star=Pi_star,
        X_train=X_tr_t,
        tau=NSC_TAU,
        deltas=deltas,
    )

    Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
    Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

    head = TabPFN25Head(head_cfg)
    head.fit(Z_tr, y_tr)

    P = head.predict_proba(Z_va)
    pred = np.argmax(P, axis=1)

    acc = accuracy_score(y_va, pred)
    f1 = f1_score(y_va, pred, average="macro")
    auc = roc_auc_score(y_va, P[:, 1])

    accs.append(acc)
    f1s.append(f1)
    aucs.append(auc)

    print(f"Fold {fold_id:02d}: ACC={acc:.4f}, F1={f1:.4f}, AUC={auc:.4f}")


print("\nFinal 5x5 CV results")
print(f"Accuracy : {np.mean(accs):.4f} ± {np.std(accs, ddof=1):.4f}")
print(f"Macro-F1 : {np.mean(f1s):.4f} ± {np.std(f1s, ddof=1):.4f}")
print(f"AUC      : {np.mean(aucs):.4f} ± {np.std(aucs, ddof=1):.4f}")

# orlaws10P multiclass classification

In [ ]:
import gc
import time
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config

# -----------------------
# User settings
# -----------------------
DATA_FILE = "orlraws10P.csv" # change this to your dataset file name
TARGET_COL = "label" # change this to your dataset target column
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Fixed GOTabPFN hyperparameters
FIXED_PARAMS = {
    "go_metric": "cosine",
    "go_num_clusters": 5,
    "go_refine_passes": 1,
    "go_direction_select": False,
    "go_feat_subsample": 3000,

    "nsc_segmentation": "uniform",
    "nsc_m_rule": "default",
    "nsc_tau": 0.99,
    "nsc_gamma": 2.049512863264476,
    "nsc_beta": 0.3887505167779042,
    "nsc_Mmin": 32,
    "nsc_Mmax": 384,
    "nsc_lmin": 12,
    "assume_standardized": False,

    "tabpfn_seed": 42,
}

# -----------------------
# Utilities
# -----------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def safe_multiclass_macro_ovr_auc(y_true, proba, num_classes):
    try:
        y_bin = label_binarize(y_true, classes=np.arange(num_classes))
        return float(
            roc_auc_score(
                y_bin,
                proba,
                average="macro",
                multi_class="ovr",
            )
        )
    except Exception:
        return np.nan


# -----------------------
# Load and preprocess data
# -----------------------
seed_everything(SEED)

df = pd.read_csv(DATA_FILE)

y_raw = df[TARGET_COL].astype(str).fillna("missing_target")
X_df = df.drop(columns=[TARGET_COL])

# Keep numeric features only
X_df = X_df.select_dtypes(include=[np.number])
X_df = X_df.apply(pd.to_numeric, errors="coerce")
X_df = X_df.fillna(X_df.median(numeric_only=True)).fillna(0.0)

# Encode multiclass labels
le = LabelEncoder()
y = le.fit_transform(y_raw).astype(np.int64)
num_classes = len(le.classes_)

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X_df.values).astype(np.float32)

print(f"X shape: {X.shape}, classes: {num_classes}")

# -----------------------
# Learn GO-LR ordering once
# -----------------------
m_full = X.shape[1]
feat_subsample = FIXED_PARAMS["go_feat_subsample"]

rng = np.random.default_rng(SEED + 999)

if feat_subsample is not None and feat_subsample < m_full:
    feat_idx = rng.choice(m_full, size=feat_subsample, replace=False)
    feat_idx.sort()
else:
    feat_idx = np.arange(m_full)

X_go = X[:, feat_idx]

go = GraphFeatureOrdering(
    num_clusters=FIXED_PARAMS["go_num_clusters"],
    metric=FIXED_PARAMS["go_metric"],
    refine=True,
    direction_select=FIXED_PARAMS["go_direction_select"],
    refine_passes=FIXED_PARAMS["go_refine_passes"],
)

try:
    Pi_sub, _, _, _ = go.fit(
        X_go,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=False,
    )
except Exception:
    cleanup_cuda()
    Pi_sub, _, _, _ = go.fit(
        X_go,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=True,
    )

ordered_subset = feat_idx[np.array(Pi_sub, dtype=np.int64)].tolist()

if len(feat_idx) < m_full:
    remaining = np.setdiff1d(np.arange(m_full), feat_idx, assume_unique=False)
    Pi_star = ordered_subset + remaining.tolist()
else:
    Pi_star = ordered_subset

Pi_star = list(map(int, Pi_star))

# -----------------------
# 5x5 cross-validation
# -----------------------
rkf = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=SEED,
)

head_cfg = TabPFN25Config(
    task_type="multiclass",
    num_classes=int(num_classes),
    device=DEVICE,
    random_state=int(FIXED_PARAMS["tabpfn_seed"]),
)

accs, f1s, aucs = [], [], []
t0 = time.perf_counter()

for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X, y), start=1):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    nsc = pidf_segpca(
        segmentation=FIXED_PARAMS["nsc_segmentation"],
        l_min=int(FIXED_PARAMS["nsc_lmin"]),
        m_rule=FIXED_PARAMS["nsc_m_rule"],
        gamma=float(FIXED_PARAMS["nsc_gamma"]),
        beta=float(FIXED_PARAMS["nsc_beta"]),
        tau=float(FIXED_PARAMS["nsc_tau"]),
        M_min=int(FIXED_PARAMS["nsc_Mmin"]),
        M_max=int(FIXED_PARAMS["nsc_Mmax"]),
        assume_standardized=bool(FIXED_PARAMS["assume_standardized"]),
        device=DEVICE,
    )

    X_tr_t = torch.from_numpy(X_tr)

    nsc.configure(
        Pi_star=Pi_star,
        X_train=X_tr_t,
        tau=float(FIXED_PARAMS["nsc_tau"]),
        deltas=None,
    )

    Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
    Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

    head = TabPFN25Head(head_cfg)
    head.fit(Z_tr, y_tr)

    P = head.predict_proba(Z_va)
    pred = np.argmax(P, axis=1)

    acc = float(accuracy_score(y_va, pred))
    f1m = float(f1_score(y_va, pred, average="macro"))
    aucm = safe_multiclass_macro_ovr_auc(y_va, P, num_classes)

    accs.append(acc)
    f1s.append(f1m)
    aucs.append(aucm)

    print(
        f"Fold {fold_id:02d}: "
        f"ACC={acc:.4f}, Macro-F1={f1m:.4f}, Macro-OvR-AUC={aucm:.4f}"
    )

    cleanup_cuda()

print("\nFinal 5x5 CV results")
print(f"Accuracy      : {np.mean(accs):.4f} ± {np.std(accs, ddof=1):.4f}")
print(f"Macro-F1      : {np.mean(f1s):.4f} ± {np.std(f1s, ddof=1):.4f}")
print(f"Macro-OvR-AUC : {np.nanmean(aucs):.4f} ± {np.nanstd(aucs, ddof=1):.4f}")
print(f"Elapsed time  : {time.perf_counter() - t0:.2f} seconds")

# DrivFace Regression example

In [ ]:
import gc
import time
import random
import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config

# -----------------------
# User settings
# -----------------------
DATA_FILE = "drivface.csv" # change this to your dataset file name
TARGET_COL = "angle" # change this to your dataset file name
SEED = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Fixed GOTabPFN hyperparameters
FIXED_PARAMS = {
    "go_metric": "manhattan",
    "go_num_clusters": 5,
    "go_refine_passes": 1,
    "go_direction_select": False,
    "go_feat_subsample": 2000,

    "nsc_segmentation": "largest_jump",
    "nsc_m_rule": "idf",
    "nsc_tau": 0.99,
    "nsc_gamma": 2.654390393837633,
    "nsc_beta": 0.043192175152615336,
    "nsc_Mmin": 16,
    "nsc_Mmax": 256,
    "nsc_lmin": 12,
    "assume_standardized": True,

    "tabpfn_seed": 3,
}

# -----------------------
# Utilities
# -----------------------
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def compute_deltas_adjacent_corr(X_tr, Pi_star, eps=1e-12):
    X_t = torch.from_numpy(X_tr).float()
    perm = torch.tensor(Pi_star, dtype=torch.long)

    Xp = X_t[:, perm]
    Xc = Xp - Xp.mean(dim=0, keepdim=True)
    std = Xc.std(dim=0, unbiased=False, keepdim=True).clamp_min(eps)
    Z = Xc / std

    corr = (Z[:, :-1] * Z[:, 1:]).mean(dim=0)
    return (1.0 - corr.abs()).cpu()


# -----------------------
# Load and preprocess data
# -----------------------
seed_everything(SEED)

df = pd.read_csv(DATA_FILE)

# Regression target
y_raw = pd.to_numeric(df[TARGET_COL], errors="coerce")
y_raw = y_raw.fillna(y_raw.median())
y = y_raw.to_numpy(dtype=np.float32)

X_df = df.drop(columns=[TARGET_COL])

# Keep numeric features only
X_df = X_df.select_dtypes(include=[np.number])
X_df = X_df.apply(pd.to_numeric, errors="coerce")
X_df = X_df.fillna(X_df.median(numeric_only=True)).fillna(0.0)

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X_df.values).astype(np.float32)

print(f"X shape: {X.shape}, y shape: {y.shape}")

# -----------------------
# Learn GO-LR ordering once
# -----------------------
m_full = X.shape[1]
feat_subsample = FIXED_PARAMS["go_feat_subsample"]

rng = np.random.default_rng(SEED + 999)

if feat_subsample is not None and feat_subsample < m_full:
    feat_idx = rng.choice(m_full, size=feat_subsample, replace=False)
    feat_idx.sort()
else:
    feat_idx = np.arange(m_full)

X_go = X[:, feat_idx]

go = GraphFeatureOrdering(
    num_clusters=FIXED_PARAMS["go_num_clusters"],
    metric=FIXED_PARAMS["go_metric"],
    refine=True,
    direction_select=FIXED_PARAMS["go_direction_select"],
    refine_passes=FIXED_PARAMS["go_refine_passes"],
)

try:
    Pi_sub, _, _, _ = go.fit(
        X_go,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=False,
    )
except Exception:
    cleanup_cuda()
    Pi_sub, _, _, _ = go.fit(
        X_go,
        seed=SEED,
        deterministic=True,
        use_cpu_kmeans=True,
    )

ordered_subset = feat_idx[np.array(Pi_sub, dtype=np.int64)].tolist()

if len(feat_idx) < m_full:
    remaining = np.setdiff1d(np.arange(m_full), feat_idx, assume_unique=False)
    Pi_star = ordered_subset + remaining.tolist()
else:
    Pi_star = ordered_subset

Pi_star = list(map(int, Pi_star))

# -----------------------
# 5x5 cross-validation
# -----------------------
rkf = RepeatedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=SEED,
)

head_cfg = TabPFN25Config(
    task_type="regression",
    num_classes=1,
    device=DEVICE,
    random_state=int(FIXED_PARAMS["tabpfn_seed"]),
)

r2s, rmses, maes = [], [], []
t0 = time.perf_counter()

for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X), start=1):
    X_tr, X_va = X[tr_idx], X[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    nsc = pidf_segpca(
        segmentation=FIXED_PARAMS["nsc_segmentation"],
        l_min=int(FIXED_PARAMS["nsc_lmin"]),
        m_rule=FIXED_PARAMS["nsc_m_rule"],
        gamma=float(FIXED_PARAMS["nsc_gamma"]),
        beta=float(FIXED_PARAMS["nsc_beta"]),
        tau=float(FIXED_PARAMS["nsc_tau"]),
        M_min=int(FIXED_PARAMS["nsc_Mmin"]),
        M_max=int(FIXED_PARAMS["nsc_Mmax"]),
        assume_standardized=bool(FIXED_PARAMS["assume_standardized"]),
        device=DEVICE,
    )

    deltas = None
    if FIXED_PARAMS["nsc_segmentation"] != "uniform":
        deltas = compute_deltas_adjacent_corr(X_tr, Pi_star)

    X_tr_t = torch.from_numpy(X_tr)

    nsc.configure(
        Pi_star=Pi_star,
        X_train=X_tr_t,
        tau=float(FIXED_PARAMS["nsc_tau"]),
        deltas=deltas,
    )

    Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
    Z_va = nsc.compress(torch.from_numpy(X_va), mode="flatten").cpu().numpy()

    head = TabPFN25Head(head_cfg)
    head.fit(Z_tr, y_tr)

    pred = np.asarray(head.predict(Z_va), dtype=np.float32).reshape(-1)

    r2 = float(r2_score(y_va, pred))
    rmse = float(np.sqrt(mean_squared_error(y_va, pred)))
    mae = float(mean_absolute_error(y_va, pred))

    r2s.append(r2)
    rmses.append(rmse)
    maes.append(mae)

    print(f"Fold {fold_id:02d}: R2={r2:.4f}, RMSE={rmse:.4f}, MAE={mae:.4f}")

    cleanup_cuda()

print("\nFinal 5x5 CV results")
print(f"R2   : {np.mean(r2s):.4f} ± {np.std(r2s, ddof=1):.4f}")
print(f"RMSE : {np.mean(rmses):.4f} ± {np.std(rmses, ddof=1):.4f}")
print(f"MAE  : {np.mean(maes):.4f} ± {np.std(maes, ddof=1):.4f}")
print(f"Elapsed time: {time.perf_counter() - t0:.2f} seconds")

# GO-LR as ordering metaheuristic on Colon

In [ ]:
# ============================================================
# Single-cell notebook test for GO-LR through current gotabpfn package
# Tests ordering runtime, TSP path cost, MinLA cost, and reordered features
#
# This code is tested on RTX A6000 GPU, which may differ from the
# original device used for paper experiments. A faster runtime may be expected.
# ============================================================

import os
import sys
import warnings
import importlib
import pandas as pd

warnings.filterwarnings(
    "ignore",
    message=".*pynvml package is deprecated.*",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    message=".*cumsum_cuda_kernel does not have a deterministic implementation.*",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message=".*Deterministic behavior was enabled.*CuBLAS.*",
    category=UserWarning,
)

# ------------------------------------------------------------
# Make current folder importable
# ------------------------------------------------------------
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# ------------------------------------------------------------
# Import package
# ------------------------------------------------------------
import gotabpfn
importlib.reload(gotabpfn)

print("[OK] Imported gotabpfn package.")

# ------------------------------------------------------------
# Config: same GO-LR settings from Colon ablation
# ------------------------------------------------------------
SEED = 42
DATA_FILE = "coloncancer_encoded.csv"
TARGET_COL = "label"
DATASET_NAME = "Colon"

BEST_GO = {
    "metric": "euclidean",
    "num_clusters": 10,
    "refine_passes": 3,
    "direction_select": True,
}

OUT_PREFIX = "colon_golr_test"

# ------------------------------------------------------------
# Check files / exports
# ------------------------------------------------------------
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"Dataset not found: {DATA_FILE}")

if getattr(gotabpfn, "run_golr_csv", None) is None:
    raise ImportError("gotabpfn.run_golr_csv is not available. Check __init__.py and GO-LR.py.")

print(f"[OK] Found dataset: {DATA_FILE}")
print("[OK] Found gotabpfn.run_golr_csv")

# ------------------------------------------------------------
# Run GO-LR
# ------------------------------------------------------------
result = gotabpfn.run_golr_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    dataset_name=DATASET_NAME,
    metric=BEST_GO["metric"],
    num_clusters=BEST_GO["num_clusters"],
    refine=True,
    direction_select=BEST_GO["direction_select"],
    refine_passes=BEST_GO["refine_passes"],
    bins=32,
    seed=SEED,
    standardize=True,
    drop_non_numeric=True,
    use_cpu_kmeans=True,   # safer for notebook testing
    save_outputs=True,
    out_prefix=OUT_PREFIX,
)

# ------------------------------------------------------------
# Display metrics
# ------------------------------------------------------------
metrics = result["metrics"]
metrics_df = pd.DataFrame([metrics])

print("\n[GO-LR metrics]")
display(metrics_df)

# ------------------------------------------------------------
# Display learned ordering preview
# ------------------------------------------------------------
ordering_df = result["ordering_df"]
print("\n[Ordering preview]")
display(ordering_df.head(20))

# ------------------------------------------------------------
# Display reordered feature table preview
# ------------------------------------------------------------
reordered_df = result["reordered_df"]
print("\n[Reordered feature table preview]")
display(reordered_df.head())

# ------------------------------------------------------------
# Access important values directly
# ------------------------------------------------------------
Pi_star = result["ordering"]
runtime_sec = result["runtime_sec"]
tsp_cost = result["tsp_path_cost"]
minla_cost = result["minla_cost"]

print("\n[Direct values]")
print(f"Number of ordered features: {len(Pi_star)}")
print(f"Runtime seconds: {runtime_sec:.6f}")
print(f"TSP path cost: {tsp_cost:.6f}")
print(f"MinLA cost: {minla_cost:.6f}")

print("\n[SAVED]")
print(f"  - {OUT_PREFIX}_reordered.csv")
print(f"  - {OUT_PREFIX}_ordering.csv")
print(f"  - {OUT_PREFIX}_metrics.json")

# Compression check

In [ ]:
# ============================================================
# Test all 4 NSC variants on Colon dataset
# Tests:
#   NSC-pSP
#   NSC-SP
#   NSC-P
#   NSC
# ============================================================

import os
import sys
import gc
import warnings
import importlib

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

warnings.filterwarnings(
    "ignore",
    message=".*pynvml package is deprecated.*",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    message=".*cumsum_cuda_kernel does not have a deterministic implementation.*",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message=".*Deterministic behavior was enabled.*CuBLAS.*",
    category=UserWarning,
)

import numpy as np
import pandas as pd
import torch

# ------------------------------------------------------------
# Make current folder importable
# ------------------------------------------------------------
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# ------------------------------------------------------------
# Import package
# ------------------------------------------------------------
import gotabpfn
importlib.reload(gotabpfn)

print("[OK] Imported gotabpfn package.")

# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
DATA_FILE = "coloncancer_encoded.csv"
TARGET_COL = "label"
DATASET_NAME = "Colon"

ORDERING_CSV = "colon_golr_ordering.csv"
SEED = 42

NSC_COMMON = {
    "segmentation": "equal_mass",
    "m_rule": "idf",
    "gamma": 1.7570143129240916,
    "beta": 0.2244046472232107,
    "M_min": 64,
    "M_max": 384,
    "l_min": 16,
    "standardize_input": True,
    "drop_non_numeric": True,
    "mode": "flatten",
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "save_outputs": True,
}

DESC_COMMON = {
    "descriptor": "basic",
    "pooling": "learn_free",
}

TAU = 0.99

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()


def maybe_make_golr_ordering():
    if os.path.exists(ORDERING_CSV):
        print(f"[OK] Found ordering file: {ORDERING_CSV}")
        return ORDERING_CSV

    if getattr(gotabpfn, "run_golr_csv", None) is None:
        print("[WARN] gotabpfn.run_golr_csv is unavailable.")
        print("[WARN] Falling back to identity ordering for all NSC variants.")
        return None

    print(f"[INFO] {ORDERING_CSV} not found. Running GO-LR through gotabpfn package...")

    gotabpfn.run_golr_csv(
        csv_path=DATA_FILE,
        target_col=TARGET_COL,
        dataset_name=DATASET_NAME,
        metric="euclidean",
        num_clusters=10,
        refine=True,
        direction_select=True,
        refine_passes=3,
        bins=32,
        seed=SEED,
        standardize=True,
        drop_non_numeric=True,
        use_cpu_kmeans=True,
        save_outputs=True,
        out_prefix="colon_golr",
    )

    if os.path.exists(ORDERING_CSV):
        print(f"[OK] Created ordering file: {ORDERING_CSV}")
        return ORDERING_CSV

    print("[WARN] GO-LR ran, but ordering file was not found.")
    print("[WARN] Falling back to identity ordering for all NSC variants.")
    return None


# ------------------------------------------------------------
# Check dataset and package exports
# ------------------------------------------------------------
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"Missing dataset file: {DATA_FILE}")

required_exports = [
    "run_nsc_psp_csv",
    "run_nsc_sp_csv",
    "run_nsc_p_csv",
    "run_nsc_csv",
]

missing_exports = [x for x in required_exports if getattr(gotabpfn, x, None) is None]
if missing_exports:
    raise ImportError(
        "Missing required gotabpfn exports:\n"
        + "\n".join([f"  - {x}" for x in missing_exports])
    )

print(f"[OK] Found dataset: {DATA_FILE}")
print("[OK] Found all NSC wrapper exports.")

ordering_csv = maybe_make_golr_ordering()

# ------------------------------------------------------------
# Run all four variants
# ------------------------------------------------------------
results = {}

print("\n" + "=" * 80)
print("Running NSC-pSP")
print("=" * 80)

results["NSC-pSP"] = gotabpfn.run_nsc_psp_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    ordering_csv=ordering_csv,
    dataset_name=DATASET_NAME,
    tau=TAU,
    out_prefix="colon_nsc_psp_test",
    **NSC_COMMON,
)

M_ref = int(results["NSC-pSP"]["metrics"]["M_selected"])
d_hat_ref = results["NSC-pSP"]["metrics"].get("d_hat_pca", None)

print(f"\n[REFERENCE from NSC-pSP] M_ref={M_ref}, d_hat_ref={d_hat_ref}")

cleanup()

print("\n" + "=" * 80)
print("Running NSC-SP")
print("=" * 80)

results["NSC-SP"] = gotabpfn.run_nsc_sp_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    ordering_csv=ordering_csv,
    dataset_name=DATASET_NAME,
    M=M_ref,
    out_prefix="colon_nsc_sp_test",
    **NSC_COMMON,
)

cleanup()

print("\n" + "=" * 80)
print("Running NSC-P")
print("=" * 80)

results["NSC-P"] = gotabpfn.run_nsc_p_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    ordering_csv=ordering_csv,
    dataset_name=DATASET_NAME,
    tau=TAU,
    out_prefix="colon_nsc_p_test",
    **NSC_COMMON,
    **DESC_COMMON,
)

cleanup()

print("\n" + "=" * 80)
print("Running NSC")
print("=" * 80)

results["NSC"] = gotabpfn.run_nsc_csv(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    ordering_csv=ordering_csv,
    dataset_name=DATASET_NAME,
    M=M_ref,
    estimate_d_hat=False,
    out_prefix="colon_nsc_test",
    **NSC_COMMON,
    **DESC_COMMON,
)

cleanup()

# ------------------------------------------------------------
# Summary table
# ------------------------------------------------------------
summary_rows = []

for name, res in results.items():
    m = res["metrics"].copy()
    m["variant"] = name
    summary_rows.append(m)

summary_df = pd.DataFrame(summary_rows)

preferred_cols = [
    "variant",
    "dataset",
    "n",
    "m_original",
    "m_compressed",
    "compression_ratio",
    "M_selected",
    "idf",
    "d_hat_pca",
    "segmentation",
    "m_rule",
    "descriptor",
    "pooling",
    "runtime_sec",
    "ordering_source",
]

available_cols = [c for c in preferred_cols if c in summary_df.columns]
remaining_cols = [c for c in summary_df.columns if c not in available_cols]
summary_df = summary_df[available_cols + remaining_cols]

print("\n" + "=" * 80)
print("FINAL SUMMARY")
print("=" * 80)

display(
    summary_df.style.format({
        "compression_ratio": "{:.4g}",
        "idf": "{:.4g}",
        "d_hat_pca": "{:.4g}",
        "runtime_sec": "{:.4g}",
    }, na_rep="NA")
)

summary_df.to_csv("colon_all_nsc_variants_summary.csv", index=False)

# ------------------------------------------------------------
# Preview compressed outputs
# ------------------------------------------------------------
for name, res in results.items():
    print(f"\n[{name}] compressed_df preview:")
    display(res["compressed_df"].head())

print("\n[SAVED SUMMARY]")
print("  - colon_all_nsc_variants_summary.csv")

print("\n[SAVED COMPRESSED FILES]")
print("  - colon_nsc_psp_test_compressed.csv")
print("  - colon_nsc_sp_test_compressed.csv")
print("  - colon_nsc_p_test_compressed.csv")
print("  - colon_nsc_test_compressed.csv")

print("\n[SAVED SEGMENTS/METRICS]")
print("  - *_segments.csv")
print("  - *_metrics.json")

# Multiple-datasets ordering diagnostics

In [ ]:
# This script:
#   1. Creates dummy high-dimensional CSV datasets.
#   2. Loads GOTabPFN's diagnostics module.
#   3. Runs diagnostics across multiple datasets.
#   4. Saves full and selected diagnostic tables.
#
# To use your own datasets, replace the `datasets` list below.
# ============================================================

import os
import sys
import gc
import random
import warnings
import importlib

warnings.filterwarnings("ignore")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# Make current folder importable, useful for local notebooks
# ------------------------------------------------------------
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


# ------------------------------------------------------------
# Import gotabpfn and diagnostics module
# ------------------------------------------------------------
import gotabpfn
importlib.reload(gotabpfn)

diag = gotabpfn.load_dataset_diagnostics_module()

print("[OK] Imported gotabpfn package.")
print("[OK] Loaded dataset diagnostics module.")


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)


# ------------------------------------------------------------
# Create dummy high-dimensional datasets
# ------------------------------------------------------------
def create_dummy_csv(
    csv_path,
    target_col,
    n_samples,
    n_features,
    n_classes,
    n_informative,
    n_redundant,
    seed,
):
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=n_informative,
        n_redundant=n_redundant,
        n_repeated=0,
        n_classes=n_classes,
        n_clusters_per_class=1,
        class_sep=1.3,
        flip_y=0.02,
        random_state=seed,
    )

    feature_cols = [f"f{i}" for i in range(n_features)]
    df = pd.DataFrame(X, columns=feature_cols)
    df[target_col] = y

    # Add one non-numeric column to show that diagnostics can drop it safely.
    df["non_numeric_id"] = [f"id_{i}" for i in range(n_samples)]

    df.to_csv(csv_path, index=False)
    print(f"[CREATED] {csv_path}: n={n_samples}, m={n_features}, classes={n_classes}")


# Dummy datasets for immediate testing.
# You can delete this section when using your own CSV files.
create_dummy_csv(
    csv_path="dummy_hdlss_1.csv",
    target_col="label",
    n_samples=80,
    n_features=2000,
    n_classes=2,
    n_informative=80,
    n_redundant=40,
    seed=SEED + 1,
)

create_dummy_csv(
    csv_path="dummy_hdlss_2.csv",
    target_col="label",
    n_samples=120,
    n_features=5000,
    n_classes=3,
    n_informative=150,
    n_redundant=80,
    seed=SEED + 2,
)

create_dummy_csv(
    csv_path="dummy_text_like.csv",
    target_col="label",
    n_samples=500,
    n_features=3000,
    n_classes=2,
    n_informative=120,
    n_redundant=100,
    seed=SEED + 3,
)

create_dummy_csv(
    csv_path="dummy_image_feature_like.csv",
    target_col="label",
    n_samples=1000,
    n_features=2048,
    n_classes=5,
    n_informative=200,
    n_redundant=100,
    seed=SEED + 4,
)


# ------------------------------------------------------------
# Patch diagnostics loader:
# Drop target column, then keep numeric feature columns only.
# ------------------------------------------------------------
def load_numeric_csv_drop_non_numeric(
    csv_path,
    target_col=None,
    standardize=True,
):
    df = pd.read_csv(csv_path)

    target_col = diag._none_if_empty(target_col)

    if target_col is not None:
        if target_col not in df.columns:
            raise ValueError(
                f"Target column '{target_col}' not found in {csv_path}.\n"
                f"Available columns: {list(df.columns)}"
            )
        df = df.drop(columns=[target_col])

    numeric_cols = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c])
    ]

    dropped_cols = [
        c for c in df.columns
        if c not in numeric_cols
    ]

    if dropped_cols:
        print(f"[INFO] {csv_path}: dropped non-numeric columns: {dropped_cols}")

    if len(numeric_cols) < 2:
        raise ValueError(
            f"{csv_path}: expected at least two numeric feature columns after "
            f"dropping target/non-numeric columns. Found {len(numeric_cols)}."
        )

    X = df[numeric_cols].to_numpy(dtype=np.float32)

    X = np.nan_to_num(
        X,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    ).astype(np.float32)

    if standardize:
        X = StandardScaler().fit_transform(X).astype(np.float32)

    return X


# Replace original loader inside diagnostics module.
diag.load_numeric_csv = load_numeric_csv_drop_non_numeric


# ------------------------------------------------------------
# Dataset list
# ------------------------------------------------------------
# CHANGE THIS BLOCK FOR YOUR OWN DATASETS.
#
# Each entry should contain:
#   csv_path     : path to the CSV file
#   target_col   : target column to remove before diagnostics
#   dataset_name : display name in the output table
#
# Example for real datasets:
#
# datasets = [
#     {"csv_path": "cellcycle.csv", "target_col": "phase", "dataset_name": "Cell Cycle"},
#     {"csv_path": "BASEHOCK.csv", "target_col": "label", "dataset_name": "BASEHOCK"},
#     {"csv_path": "RELATHE.csv", "target_col": "label", "dataset_name": "RELATHE"},
#     {"csv_path": "PCMAC.csv", "target_col": "label", "dataset_name": "PCMAC"},
#     {"csv_path": "orlraws10P.csv", "target_col": "label", "dataset_name": "orlraws10P"},
# ]
datasets = [
    {
        "csv_path": "dummy_hdlss_1.csv",
        "target_col": "label",
        "dataset_name": "Dummy-HDLSS-1",
    },
    {
        "csv_path": "dummy_hdlss_2.csv",
        "target_col": "label",
        "dataset_name": "Dummy-HDLSS-2",
    },
    {
        "csv_path": "dummy_text_like.csv",
        "target_col": "label",
        "dataset_name": "Dummy-Text-Like",
    },
    {
        "csv_path": "dummy_image_feature_like.csv",
        "target_col": "label",
        "dataset_name": "Dummy-Image-Feature-Like",
    },
]


# ------------------------------------------------------------
# Check files
# ------------------------------------------------------------
missing = [d["csv_path"] for d in datasets if not os.path.exists(d["csv_path"])]

if missing:
    raise FileNotFoundError(
        "Missing dataset file(s):\n" + "\n".join([f"  - {f}" for f in missing])
    )

print("[OK] All dataset files found.")


# ------------------------------------------------------------
# Run diagnostics
# ------------------------------------------------------------
df_metrics = diag.analyze_many_csvs(
    datasets=datasets,
    out_csv="multi_dataset_ordering_metrics.csv",
    standardize=True,
    verbose=True,
)


# ------------------------------------------------------------
# Select final columns
# ------------------------------------------------------------
show_cols = [
    "FOE_rank",
    "dataset",
    "category",
    "n",
    "m",
    "rho",
    "IDF_final",
    "FOE",
    "P_success",
    "Delta_AdjCoh",
    "Delta_HitRate",
    "Delta_Cut",
    "LES",
    "AUC",
]

available_cols = [c for c in show_cols if c in df_metrics.columns]
df_show = df_metrics[available_cols].copy()

# Save selected table
df_show.to_csv("multi_dataset_ordering_metrics_selected_columns.csv", index=False)

print("\n[SAVED]")
print("  - multi_dataset_ordering_metrics.csv")
print("  - multi_dataset_ordering_metrics_selected_columns.csv")


# ------------------------------------------------------------
# Pretty display
# ------------------------------------------------------------
format_dict = {
    "rho": "{:.4g}",
    "IDF_final": "{:.4g}",
    "FOE": "{:.4g}",
    "P_success": "{:.4g}",
    "Delta_AdjCoh": "{:.4g}",
    "Delta_HitRate": "{:.4g}",
    "Delta_Cut": "{:.4g}",
    "LES": "{:.4g}",
    "AUC": "{:.4g}",
}

try:
    display(
        df_show.style.format(
            {k: v for k, v in format_dict.items() if k in df_show.columns}
        )
    )
except NameError:
    print(df_show)

# Single dataset ordering diagnostics

In [ ]:
# ============================================================
# This script:
#   1. Creates one dummy high-dimensional CSV dataset.
#   2. Loads GOTabPFN's diagnostics module.
#   3. Runs ordering diagnostics for one dataset.
#   4. Saves full and selected diagnostic tables.
#
# To use your own dataset, change DATA_FILE, TARGET_COL, and DATASET_NAME.
# ============================================================

import os
import sys
import random
import warnings
import importlib

warnings.filterwarnings("ignore")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------
SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)

seed_everything(SEED)


# ------------------------------------------------------------
# Optional: create a dummy high-dimensional dataset
# ------------------------------------------------------------
# You can delete this section when using your own CSV file.
def create_dummy_single_dataset(
    csv_path,
    target_col,
    n_samples=120,
    n_features=3000,
    n_classes=3,
    n_informative=150,
    n_redundant=80,
    seed=42,
):
    X, y = make_classification(
        n_samples=n_samples,
        n_features=n_features,
        n_informative=n_informative,
        n_redundant=n_redundant,
        n_repeated=0,
        n_classes=n_classes,
        n_clusters_per_class=1,
        class_sep=1.3,
        flip_y=0.02,
        random_state=seed,
    )

    feature_cols = [f"f{i}" for i in range(n_features)]
    df = pd.DataFrame(X, columns=feature_cols)
    df[target_col] = y

    # Add one non-numeric column to show that diagnostics can drop it safely.
    df["non_numeric_id"] = [f"id_{i}" for i in range(n_samples)]

    df.to_csv(csv_path, index=False)
    print(f"[CREATED] {csv_path}: n={n_samples}, m={n_features}, classes={n_classes}")


# ------------------------------------------------------------
# User input: change these three lines for your own dataset
# ------------------------------------------------------------
DATA_FILE = "dummy_single_diagnostics.csv"
TARGET_COL = "label"
DATASET_NAME = "Dummy Single Dataset"

# Create dummy CSV for immediate testing.
# Comment this out when using your own existing CSV file.
create_dummy_single_dataset(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    n_samples=120,
    n_features=3000,
    n_classes=3,
    n_informative=150,
    n_redundant=80,
    seed=SEED,
)

OUT_CSV = f"{DATASET_NAME.replace(' ', '_').replace('/', '_')}_single_ordering_metrics.csv"


# ------------------------------------------------------------
# Make current folder importable, useful for local notebooks
# ------------------------------------------------------------
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())


# ------------------------------------------------------------
# Import gotabpfn and diagnostics module
# ------------------------------------------------------------
import gotabpfn
importlib.reload(gotabpfn)

diag = gotabpfn.load_dataset_diagnostics_module()

print("[OK] Imported gotabpfn package.")
print("[OK] Loaded dataset diagnostics module.")


# ------------------------------------------------------------
# Patch diagnostics loader:
# Drop target column, then keep numeric feature columns only.
# ------------------------------------------------------------
def load_numeric_csv_drop_non_numeric(
    csv_path,
    target_col=None,
    standardize=True,
):
    df = pd.read_csv(csv_path)

    target_col = diag._none_if_empty(target_col)

    if target_col is not None:
        if target_col not in df.columns:
            raise ValueError(
                f"Target column '{target_col}' not found in {csv_path}.\n"
                f"Available columns: {list(df.columns)}"
            )
        df = df.drop(columns=[target_col])

    numeric_cols = [
        c for c in df.columns
        if pd.api.types.is_numeric_dtype(df[c])
    ]

    dropped_cols = [
        c for c in df.columns
        if c not in numeric_cols
    ]

    if dropped_cols:
        print(f"[INFO] {csv_path}: dropped non-numeric columns: {dropped_cols}")

    if len(numeric_cols) < 2:
        raise ValueError(
            f"{csv_path}: expected at least two numeric feature columns after "
            f"dropping target/non-numeric columns. Found {len(numeric_cols)}."
        )

    X = df[numeric_cols].to_numpy(dtype=np.float32)

    X = np.nan_to_num(
        X,
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    ).astype(np.float32)

    if standardize:
        X = StandardScaler().fit_transform(X).astype(np.float32)

    return X


# Replace original loader inside diagnostics module.
diag.load_numeric_csv = load_numeric_csv_drop_non_numeric


# ------------------------------------------------------------
# Check file
# ------------------------------------------------------------
if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f"Missing dataset file: {DATA_FILE}")

print(f"[OK] Dataset file found: {DATA_FILE}")


# ------------------------------------------------------------
# Run diagnostics for one dataset
# ------------------------------------------------------------
df_metrics = diag.analyze_csv_ordering_metrics(
    csv_path=DATA_FILE,
    target_col=TARGET_COL,
    dataset_name=DATASET_NAME,
    out_csv=OUT_CSV,
    standardize=True,
    verbose=True,
)


# ------------------------------------------------------------
# Select final columns
# ------------------------------------------------------------
show_cols = [
    "FOE_rank",
    "dataset",
    "category",
    "n",
    "m",
    "rho",
    "IDF_final",
    "FOE",
    "P_success",
    "Delta_AdjCoh",
    "Delta_HitRate",
    "Delta_Cut",
    "LES",
    "AUC",
]

available_cols = [c for c in show_cols if c in df_metrics.columns]
df_show = df_metrics[available_cols].copy()

# Save selected table
selected_csv = OUT_CSV.replace(".csv", "_selected_columns.csv")
df_show.to_csv(selected_csv, index=False)

print("\n[SAVED]")
print(f"  - {OUT_CSV}")
print(f"  - {selected_csv}")


# ------------------------------------------------------------
# Pretty display
# ------------------------------------------------------------
format_dict = {
    "rho": "{:.4g}",
    "IDF_final": "{:.4g}",
    "FOE": "{:.4g}",
    "P_success": "{:.4g}",
    "Delta_AdjCoh": "{:.4g}",
    "Delta_HitRate": "{:.4g}",
    "Delta_Cut": "{:.4g}",
    "LES": "{:.4g}",
    "AUC": "{:.4g}",
}

try:
    display(
        df_show.style.format(
            {k: v for k, v in format_dict.items() if k in df_show.columns}
        )
    )
except NameError:
    print(df_show)